# FishSizeEstimate — V12 SAFE SAVE VERSION (NO TRAINING)

**Goal:** Diagnose the existing V11 fish-instance segmenter with a short, memory-bounded Kaggle run. **No model training, no dataset upload, no centimeter claims.**

Inputs already mounted:
- `/kaggle/input/datasets/wafabila/sfishtrack-dataset` — YOLO fish segmentation images and labels
- `/kaggle/input/datasets/wafabila/output-v11` — trained V11 model and official V11 evaluation

**What changed vs. V12 draft that stopped in the middle:** no full-data polygon scan, no retention of hundreds of masks in RAM, samples only 2 frames per video in validation and test, no full COCO validation, no GPU required. Results are **sampled diagnostics only**; the V11 test mAP remains authoritative. Uses 320×320 diagnostic raster masks; thin tails and small instances may be affected. Test is previously evaluated in V11, so it is exploratory analysis, not a fresh held-out estimate.

After every code cell completes, outputs are stored under `/kaggle/working/FishSizeEstimate_V12_SAFE/`. Final cell creates `/kaggle/working/FishSizeEstimate_V12_SAFE_Results.zip` and verifies its contents. Run **Save Version → Save & Run All** with Accelerator = None (CPU) or T4, then download the ZIP.

## 0. Config and input paths

Change only `SAMPLES_PER_VIDEO` if necessary. Default is 2 per validation video and 2 per test video (typically 32 frames total). No training calls exist in this notebook.

In [ ]:
from pathlib import Path
import os, sys, json, csv, random, re, time, hashlib, shutil, zipfile, subprocess, importlib.util
from collections import defaultdict, Counter

SEED = 42
SAMPLES_PER_VIDEO = 2     # keep small for Save Version (usually 32 images)
IMGSZ = 384              # inference resolution: faster than old 512
RASTER = 320             # masks compared at 320x320 for a fast approximate audit
CANDIDATE_CONF = 0.10
MATCH_IOU = 0.50
THRESHOLDS = [0.15, 0.25, 0.40]
MAX_DET = 60             # tradeoff: crowded scenes with >60 fish may be truncated
MAX_OVERLAYS = 12

DATA_INPUT = Path('/kaggle/input/datasets/wafabila/sfishtrack-dataset')
V11_INPUT = Path('/kaggle/input/datasets/wafabila/output-v11')
OUT = Path('/kaggle/working/FishSizeEstimate_V12_SAFE')
OUT.mkdir(parents=True, exist_ok=True)

assert DATA_INPUT.exists(), f'Missing input: {DATA_INPUT}. Use Add Input.'
assert V11_INPUT.exists(), f'Missing input: {V11_INPUT}. Use Add Input.'
random.seed(SEED)
print('Python:', sys.version.split()[0]);print('Dataset:', DATA_INPUT);print('V11:',V11_INPUT)
print('NO TRAINING; samples/video/split:',SAMPLES_PER_VIDEO)

## 1. Dependencies, model and evaluation provenance

Use existing V11 weights; installing Ultralytics is only attempted if unavailable. The checkpoint is loaded only from your own Kaggle dataset. PyTorch checkpoint loading requires trusting its source.

In [ ]:
if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3,<9'])
import numpy as np, cv2, torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yaml
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if DEVICE=='cpu': torch.set_num_threads(min(4, os.cpu_count() or 2))
print('Torch:',torch.__version__, 'Device:', DEVICE)

def is_data_root(p):
    return (p/'data.yaml').is_file() and all((p/'images'/x).is_dir() and (p/'labels'/x).is_dir() for x in ('train','val','test'))
DATA_ROOT = DATA_INPUT if is_data_root(DATA_INPUT) else next((p.parent for p in DATA_INPUT.rglob('data.yaml') if is_data_root(p.parent)),None)
if DATA_ROOT is None: raise FileNotFoundError('YOLO data.yaml / images / labels not found')

weights = sorted(V11_INPUT.rglob('best_fish_segmenter.pt'))
if not weights:weights=sorted(V11_INPUT.rglob('best.pt'))
if not weights: raise FileNotFoundError('No V11 segmentation checkpoint in output-v11')
WEIGHTS = weights[0]
print('Weights:',WEIGHTS, 'MB:',round(WEIGHTS.stat().st_size/1e6,2))
MODEL = YOLO(str(WEIGHTS))
assert MODEL.task=='segment', 'Checkpoint is not a segmentation model!'

report_paths = sorted(V11_INPUT.rglob('heldout_test_report.json'))
V11_REPORT = json.loads(report_paths[0].read_text(encoding='utf-8-sig')) if report_paths else {}
(OUT/'v11_official_metrics.json').write_text(json.dumps({
    'v11_report_path':str(report_paths[0]) if report_paths else None,
    'validation_metrics':V11_REPORT.get('validation_metrics'),
    'heldout_test_metrics':V11_REPORT.get('heldout_test_metrics')
},indent=2),encoding='utf-8')
print('Existing V11 test metrics:',V11_REPORT.get('heldout_test_metrics','Not in dataset'))

## 2. Verify split groups and select deterministic samples

No random frame-level splitting. Sample validation and the **previously inspected** test set separately, by video. Model selection and threshold choice rely on validation only.

In [ ]:
IMG_EXT = {'.jpg','.jpeg','.png','.webp','.bmp'}
all_videos={};samples={}
for split in ('train','val','test'):
    group=defaultdict(list)
    for p in sorted((DATA_ROOT/'images'/split).iterdir()):
        if p.suffix.lower() not in IMG_EXT or not p.is_file():continue
        m=re.match(r'^video(\d+)_',p.stem)
        if not m:raise RuntimeError('Missing video ID: '+p.name)
        if not (DATA_ROOT/'labels'/split/(p.stem+'.txt')).is_file():raise RuntimeError('Missing annotation: '+p.name)
        group[m.group(1)].append(p)
    all_videos[split]=group
    print(split, 'video groups:',len(group),'frames:',sum(map(len,group.values())))
for a,b in [('train','val'),('train','test'),('val','test')]:
    assert not (set(all_videos[a]) & set(all_videos[b])),f'Video leakage {a} / {b}'
for split in ('val','test'):
    rng=random.Random(SEED+(0 if split=='val' else 17))
    samples[split]=[p for video,paths in sorted(all_videos[split].items()) for p in rng.sample(paths,min(SAMPLES_PER_VIDEO,len(paths)))]
    print('Sample',split,len(samples[split]))
assert samples['val'] and samples['test']
(OUT/'sample_manifest.json').write_text(json.dumps({
 'seed':SEED,'samples_per_video':SAMPLES_PER_VIDEO,
 'video_groups':{k:sorted(x) for k,x in all_videos.items()},
 'samples':{k:[p.name for p in paths] for k,paths in samples.items()}
},indent=2),encoding='utf-8')

## 3. Streaming inference and matching helpers

**Important:** Only the current frame's masks live in memory. We aggregate confusion counts immediately. The raster matching is an approximate diagnostic, **not official COCO mAP**. Each fish polygon is treated as a separate ground-truth instance; multiple predictions cannot match the same ground-truth instance.

In [ ]:
def parse_gt(p,split):
    label=DATA_ROOT/'labels'/split/(p.stem+'.txt')
    masks=[];areas=[]
    for row in label.read_text(encoding='utf-8-sig').splitlines():
        if not row.strip():continue
        a=np.fromstring(row,sep=' ')
        if len(a)<7 or len(a)%2!=1 or not np.isfinite(a).all() or a[0]!=0: raise ValueError(f'Invalid annotation {label}')
        xy=a[1:].reshape(-1,2)
        if np.any(xy<0) or np.any(xy>1):raise ValueError(f'Out of range polygon {label}')
        mask=np.zeros((RASTER,RASTER),np.uint8)
        pts=np.rint(xy*(RASTER-1)).astype(np.int32)
        cv2.fillPoly(mask,[pts],1)
        masks.append(mask.astype(bool));areas.append(float(mask.mean()))
    return masks, areas

def bucket(area):
    return 'tiny (<0.25%)' if area<.0025 else ('medium (0.25-2%)' if area<.02 else 'large (>=2%)')

def match(gt,preds):
    pairs=[]
    for i,g in enumerate(gt):
        gsum=int(g.sum())
        for j,p in enumerate(preds):
            overlap=int(np.count_nonzero(g&p));union=gsum+int(p.sum())-overlap
            iou=overlap/union if union else 0.
            if iou>=MATCH_IOU:pairs.append((iou,i,j))
    pairs.sort(reverse=True);gs=set();ps=set();matched=[]
    for iou,i,j in pairs:
        if i not in gs and j not in ps:gs.add(i);ps.add(j);matched.append((i,j,iou))
    return matched

def evaluate_single(gt,areas,confidence,masks,threshold):
    selected=[m for c,m in zip(confidence,masks) if c>=threshold]
    matched=match(gt,selected)
    matched_gt={i for i,_,_ in matched}
    return {'tp':len(matched),'fp':len(selected)-len(matched),'fn':len(gt)-len(matched),
            'gt':len(gt),'pred':len(selected),'area':Counter(bucket(a) for a in areas),
            'found':Counter(bucket(a) for i,a in enumerate(areas) if i in matched_gt)}

def totals_to_scores(t):
    tp,fp,fn=t['tp'],t['fp'],t['fn']
    p=tp/(tp+fp) if tp+fp else 0.
    r=tp/(tp+fn) if tp+fn else 0.
    return {'tp':tp,'fp':fp,'fn':fn,'precision':round(p,5),'recall':round(r,5),'f1':round(2*p*r/(p+r),5) if p+r else 0.}

def predict_one(image):
    result=MODEL.predict(source=str(image),imgsz=IMGSZ,conf=CANDIDATE_CONF,
        device=DEVICE,retina_masks=True,max_det=MAX_DET,verbose=False)[0]
    conf=[];masks=[]
    if result.boxes is not None and result.masks is not None:
        conf=result.boxes.conf.detach().cpu().numpy().tolist()
        for m in result.masks.data.detach().cpu().numpy():
            masks.append(cv2.resize((m>0.5).astype(np.uint8),(RASTER,RASTER),interpolation=cv2.INTER_NEAREST).astype(bool))
    return conf,masks

CSV_FIELDS=['split','video','image','gt','pred','tp','fp','fn','threshold','inference_seconds']
area_totals=defaultdict(Counter)
area_found=defaultdict(Counter)
video_totals=defaultdict(Counter)

## 4. Validation-only confidence sweep — bounded and resumable summaries

At most `SAMPLES_PER_VIDEO × 8` validation images are inferred. No full-result mask accumulation. A CSV row is flushed for every frame, so interrupted runs leave useful evidence. This is diagnostic threshold selection, not replacement for V11 mAP.

In [ ]:
import gc
from tqdm.auto import tqdm
val_totals={t:Counter() for t in THRESHOLDS}
val_video={t:defaultdict(Counter) for t in THRESHOLDS}
val_size_gt={t:Counter() for t in THRESHOLDS}
val_size_found={t:Counter() for t in THRESHOLDS}
failures=[]
logpath=OUT/'validation_sweep_frames.csv'
with logpath.open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=CSV_FIELDS);writer.writeheader()
    for p in tqdm(samples['val'],desc='VAL streaming diagnostics'):
        t0=time.monotonic()
        try:
            gt,areas=parse_gt(p,'val')
            conf,mask=predict_one(p)
            for t in THRESHOLDS:
                d=evaluate_single(gt,areas,conf,mask,t)
                val_totals[t].update({k:d[k] for k in ('tp','fp','fn')})
                video=p.stem.split('_')[0]
                val_video[t][video].update({k:d[k] for k in ('tp','fp','fn')})
                val_size_gt[t].update(d['area']);val_size_found[t].update(d['found'])
                writer.writerow({'split':'val','video':video,'image':p.name,
                  **{k:d[k] for k in ('tp','fp','fn','gt','pred')},
                  'threshold':t,'inference_seconds':round(time.monotonic()-t0,2)})
            f.flush()
            del gt,mask
        except Exception as e:
            failures.append({'split':'val','image':p.name,'error':str(e)})
        finally:gc.collect()

sweep={str(t):totals_to_scores(val_totals[t]) for t in THRESHOLDS}
if not any(val_totals[t]['tp']+val_totals[t]['fn'] for t in THRESHOLDS):
    raise RuntimeError('Validation produced no evaluable GT instances. See error log / mounted labels.')
SELECTED=max(THRESHOLDS,key=lambda t:(sweep[str(t)]['f1'],-t))
(OUT/'validation_threshold_sweep.json').write_text(json.dumps({
    'selected_threshold_from_val_only':SELECTED,'sampled_results_NOT_mAP':sweep,
    'failures':failures},indent=2),encoding='utf-8')
print('Selected confidence (VAL ONLY):',SELECTED)
print(json.dumps(sweep,indent=2))
print('Failed samples:',len(failures))

## 5. Test diagnostics — previously evaluated set, no tuning

Inspect at the threshold selected from validation. For 8 test videos × 2 frames, this remains small and does not change model weights. Generate up to 12 side-by-side examples during streaming rather than retaining large masks.

In [ ]:
from PIL import Image
OVERLAYS=OUT/'overlays';OVERLAYS.mkdir(exist_ok=True)
all_video=defaultdict(Counter)
all_gt=defaultdict(Counter);all_found=defaultdict(Counter)
for video,c in val_video[SELECTED].items():all_video[('val',video)].update(c)
all_gt['val'].update(val_size_gt[SELECTED]);all_found['val'].update(val_size_found[SELECTED])
test_total=Counter();saved=0

def draw_overlay(p,gt,pred,out_path):
    rgb=cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB)
    rgb=cv2.resize(rgb,(RASTER,RASTER),interpolation=cv2.INTER_AREA)
    g=np.zeros((RASTER,RASTER),bool);q=np.zeros((RASTER,RASTER),bool)
    for x in gt:g|=x
    for x in pred:q|=x
    left=rgb.copy();right=rgb.copy()
    left[g]=(left[g]*.55+np.array([0,190,0])*.45).astype(np.uint8)
    right[q]=(right[q]*.55+np.array([220,0,0])*.45).astype(np.uint8)
    view=np.concatenate([rgb,left,right],axis=1)
    Image.fromarray(view).save(out_path,quality=82)

logpath=OUT/'test_diagnostic_frames.csv'
with logpath.open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=CSV_FIELDS);writer.writeheader()
    for p in tqdm(samples['test'],desc='TEST streaming diagnostics'):
        t0=time.monotonic()
        try:
            gt,areas=parse_gt(p,'test')
            conf,mask=predict_one(p)
            selected_masks=[m for c,m in zip(conf,mask) if c>=SELECTED]
            d=evaluate_single(gt,areas,conf,mask,SELECTED)
            video=p.stem.split('_')[0]
            test_total.update({k:d[k] for k in ('tp','fp','fn')})
            all_video[('test',video)].update({k:d[k] for k in ('tp','fp','fn')})
            all_gt['test'].update(d['area']);all_found['test'].update(d['found'])
            writer.writerow({'split':'test','video':video,'image':p.name,
                **{k:d[k] for k in ('tp','fp','fn','gt','pred')},
                'threshold':SELECTED,'inference_seconds':round(time.monotonic()-t0,2)})
            f.flush()
            if saved<MAX_OVERLAYS:
                draw_overlay(p,gt,selected_masks,OVERLAYS/f'{saved:02d}_{video}.jpg');saved+=1
            del gt,mask,selected_masks
        except Exception as e:
            failures.append({'split':'test','image':p.name,'error':str(e)})
        finally:gc.collect()
print('Sampled TEST metrics (NOT mAP):',totals_to_scores(test_total))
print('Overlay count:',saved,'Failures:',len(failures))

## 6. Plots, scientific report and final ZIP

Export real measurements only. Keep the trained checkpoint in `output-v11`; **do not duplicate it inside this ZIP** unless explicitly needed (saves time). Downloadable package contains CSVs, JSONs, graphs and overlay examples. This cell always checks archive integrity. **There is no `model.train` anywhere in V12.**

In [ ]:
video_rows=[];size_rows=[]
for (split,video),t in sorted(all_video.items()):
    video_rows.append({'split':split,'video':video,**totals_to_scores(t)})
for split in ('val','test'):
    for name in ['tiny (<0.25%)','medium (0.25-2%)','large (>=2%)']:
        count=all_gt[split][name];found=all_found[split][name]
        size_rows.append({'split':split,'size':name,'gt_count':count,'matched':found,'recall':round(found/count,5) if count else None})
def csv_write(name,rows):
    if not rows:return
    with (OUT/name).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
csv_write('per_video_summary.csv',video_rows)
csv_write('per_size_summary.csv',size_rows)

fig,ax=plt.subplots(figsize=(12,5))
for split,color in [('val','tab:blue'),('test','tab:orange')]:
    rr=[r for r in video_rows if r['split']==split]
    ax.plot(range(len(rr)),[r['recall'] for r in rr],marker='o',label=split)
ax.set(xlabel='Video index within split',ylabel='Sampled mask recall @ IoU 0.5',title='Diagnostics only — few frames per video',ylim=(0,1));ax.legend();fig.tight_layout()
fig.savefig(OUT/'sampled_recall_per_video.png',dpi=120);plt.close(fig)

h=hashlib.sha256()
with WEIGHTS.open('rb') as f:
    for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
report={
    'status':'complete' if not failures else 'partial_with_failed_samples',
    'task':'streaming sampled fish instance-segmentation diagnostics; NO TRAINING',
    'sampled_validation_scores_NOT_official_mAP':sweep,
    'sampled_test_scores_NOT_official_mAP':totals_to_scores(test_total),
    'selected_confidence_on_validation':SELECTED,
    'v11_official_heldout_metrics':V11_REPORT.get('heldout_test_metrics'),
    'dataset':str(DATA_ROOT),'weights':str(WEIGHTS),'weights_sha256':h.hexdigest(),
    'settings':{'samples_per_video_per_split':SAMPLES_PER_VIDEO,'imgsz':IMGSZ,'raster':RASTER,
        'match_iou':MATCH_IOU,'max_det':MAX_DET,'device':str(DEVICE),'seed':SEED},
    'failed_samples':failures,
    'limitations':['Sampled results are not COCO mAP, do not replace V11 official test results',
    'Previously evaluated test used for exploratory diagnosis, not fresh held-out validation',
    'Masks and GT rasterized to square 320x320; thin detail and precise IoU can differ',
    'max_det may truncate very crowded scenes','Dataset conversion excluded 4174 source frames',
    'No fish length in centimeters or calibrated stereo measured']
}
(OUT/'diagnostic_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
(OUT/'README.md').write_text('V12 SAFE: sampled, streaming diagnosis only. V11 weights remain in output-v11. Not training or calibrated centimeter measurement. Read diagnostic_report.json.\n',encoding='utf-8')
ZIP=Path('/kaggle/working/FishSizeEstimate_V12_SAFE_Results.zip')
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,p.relative_to(OUT).as_posix())
with zipfile.ZipFile(ZIP) as z:
    bad=z.testzip()
    assert bad is None, f'Archive corrupt: {bad}'
    assert 'diagnostic_report.json' in z.namelist()
assert ZIP.stat().st_size>1000, 'ZIP unexpectedly tiny'
print('\nSUCCESS — SAVE VERSION READY, NO TRAINING')
print('ZIP:',ZIP,'size MiB:',round(ZIP.stat().st_size/2**20,2))
print('V11 checkpoint reused:',WEIGHTS)
print('Sampled test scores (NOT mAP):',totals_to_scores(test_total))
print('Failed samples:',len(failures))